This notebook covers advanced VPC connectivity: peering VPCs together, accessing AWS services privately with VPC endpoints and PrivateLink, scaling to many VPCs with Transit Gateway, and connecting on-premises networks to AWS via Site-to-Site VPN and Direct Connect. These are essential architectural patterns for real-world AWS deployments and appear heavily on the SAA-C03 exam.

## VPC Peering

VPC Peering creates a direct network connection between two VPCs, allowing instances in each VPC to communicate using private IP addresses as if they were on the same network.

### Key properties
- Works between VPCs in the **same account, different accounts, same region, or different regions**
- **No overlapping CIDR blocks** — if both VPCs use `10.0.0.0/16`, peering is not possible
- **Not transitive** — if VPC A peers with VPC B and VPC B peers with VPC C, traffic from A cannot reach C through B. You must create a direct A↔C peering connection.
- After accepting the peering request, you must **update route tables** in both VPCs to route traffic for the peer's CIDR to the peering connection
- **No bandwidth limit** — traffic uses AWS backbone

```
VPC A (10.0.0.0/16) ←──── peering ────→ VPC B (10.1.0.0/16)
                                                │
                                            peering
                                                │
                                          VPC C (10.2.0.0/16)

A cannot reach C — peering is NOT transitive.
To connect A↔C you need a direct A↔C peering connection.
```

### Route table requirement
In VPC A's route table: add route `10.1.0.0/16 → pcx-xxx` (peering connection)
In VPC B's route table: add route `10.0.0.0/16 → pcx-xxx`

Security groups can reference security groups from a peered VPC (cross-account SG referencing is supported in the same region).

> **When to use peering**: small number of VPCs (2–4) that need to communicate. For many VPCs, use Transit Gateway instead.

## VPC Endpoints

VPC Endpoints let you access AWS services **privately from within your VPC** — without traffic leaving the AWS network, without an Internet Gateway, NAT Gateway, or VPN.

### Two types

#### Gateway Endpoints
- Available only for **S3** and **DynamoDB**
- **Free** — no hourly charge
- Work by adding a **route table entry** that points traffic destined for S3/DynamoDB to the endpoint
- Traffic stays on the AWS network; the S3/DynamoDB public IP range is reached via the private route
- Scoped to a region (not a specific AZ)

```
Private EC2  →  Gateway Endpoint  →  S3 (no NAT Gateway needed)
Route table: pl-xxxxxxxx (S3 prefix list) → vpce-xxx
```

#### Interface Endpoints (powered by AWS PrivateLink)
- Available for **most AWS services** (CloudWatch, SNS, SQS, API Gateway, EC2 API, KMS, etc.)
- Create an **ENI** (Elastic Network Interface) in your subnet with a private IP
- Traffic from your VPC goes to the ENI, which routes to the AWS service privately
- Charged **per hour** + **per GB** processed
- Support **endpoint policies** to restrict which principals and actions are allowed
- Support **private DNS** — when enabled, the standard service DNS name (e.g. `sqs.us-east-1.amazonaws.com`) resolves to the private ENI IP inside your VPC

### Endpoint policies
Both endpoint types support **endpoint policies** — IAM-style resource policies that control what can be accessed through the endpoint. Example: restrict the S3 gateway endpoint to only allow access to a specific bucket.

### When to use which
| Scenario | Endpoint type |
|---|---|
| Private EC2 accessing S3 without NAT | Gateway endpoint |
| Private EC2 accessing DynamoDB without NAT | Gateway endpoint |
| Private EC2 calling CloudWatch, SQS, SSM, etc. | Interface endpoint |
| On-premises servers accessing AWS services via Direct Connect | Interface endpoint |

## AWS PrivateLink

PrivateLink lets you **expose a service in your VPC to other VPCs** (or other accounts) privately — without peering, without Internet Gateway, without overlapping CIDR concerns.

### How it works
```
Consumer VPC                    Provider VPC
┌─────────────────┐             ┌─────────────────────┐
│  Interface      │             │   NLB               │
│  Endpoint (ENI) │────────────▶│   (Network Load     │
│  private IP     │  PrivateLink│    Balancer)         │
└─────────────────┘             │   ↓                 │
                                │   Service instances │
                                └─────────────────────┘
```

- The **provider** puts a **Network Load Balancer** in front of their service
- The **consumer** creates an **Interface Endpoint** in their VPC
- Traffic flows privately over AWS backbone — CIDRs never need to match

### Use cases
- SaaS providers exposing a service to thousands of customer VPCs
- Sharing a microservice between internal teams' VPCs without full VPC peering
- All AWS Interface Endpoints are powered by PrivateLink

> **Exam tip:** PrivateLink = expose a service, not an entire VPC. VPC Peering = connect two entire VPCs. PrivateLink scales to thousands of consumers; peering does not.

## AWS Transit Gateway

Transit Gateway (TGW) is a **regional network hub** that connects multiple VPCs, VPN connections, and Direct Connect gateways through a single managed gateway.

### The problem it solves
With VPC Peering, connecting N VPCs requires N×(N-1)/2 peering connections (a full mesh). With 10 VPCs that's 45 peering connections to manage. Transit Gateway replaces this with a **hub-and-spoke** topology — each VPC connects to TGW, and TGW routes between them.

```
Without TGW (full mesh):          With TGW (hub-and-spoke):
A ─── B                           A
│ ╲ ╱ │                           │
│  ╳  │                      B ── TGW ── C
│ ╱ ╲ │                           │
C ─── D                           D
6 peering connections              4 attachments
```

### Key features
- **Transitive routing** — traffic can flow between any two attachments (unlike VPC Peering)
- Supports attachments: **VPCs, Site-to-Site VPN, Direct Connect Gateway, other TGWs**
- **TGW route tables** — control which attachments can communicate with which (route segmentation)
- **Inter-region peering** — peer TGWs across regions to build a global network
- **Multicast support**
- Bandwidth scales automatically

### Route segmentation example
Separate development and production VPCs:
```
TGW Route Table 1 (prod):   prod-vpc-A, prod-vpc-B, on-prem (DX)
TGW Route Table 2 (dev):    dev-vpc-A,  dev-vpc-B
```
Dev VPCs cannot reach prod VPCs — they're in different TGW route tables.

### RAM sharing
Transit Gateway can be **shared across accounts** using AWS Resource Access Manager (RAM). One central networking account owns the TGW; all other accounts attach their VPCs to it.

> **Exam tip:** VPC Peering for a few VPCs in the same region. Transit Gateway for many VPCs, cross-region, or when you need transitive routing.

## Site-to-Site VPN

Site-to-Site VPN creates an **IPsec encrypted tunnel** between your on-premises network and your AWS VPC over the public internet.

### Components

| Component | Description |
|---|---|
| **Virtual Private Gateway (VGW)** | AWS side of the VPN — attached to your VPC |
| **Customer Gateway (CGW)** | Represents your on-premises VPN device in AWS (IP address or FQDN) |
| **VPN Connection** | The IPsec tunnel between VGW and CGW |

### Two tunnels
Every VPN connection has **two tunnels** (two endpoints on the AWS side) for redundancy. Configure your on-premises device to use both. If one tunnel goes down, traffic fails over to the second automatically.

### Routing
- **Static routing**: you manually specify which CIDRs to route over the VPN
- **Dynamic routing (BGP)**: your on-premises router and VGW exchange routes automatically via BGP. Required for VPN CloudHub.

### VPN CloudHub
Connect **multiple on-premises sites** to a single VGW — each site can communicate with AWS and with each other through the hub. Requires BGP.

### Limitations
- Runs over the **public internet** — variable latency, lower bandwidth than Direct Connect
- Maximum **1.25 Gbps** per tunnel
- Encrypted by default (IPsec)

### VPN to Transit Gateway
Attach a VPN connection directly to a Transit Gateway instead of a VGW. This lets all VPCs attached to the TGW reach the on-premises network — no need for a VGW per VPC.

## AWS Direct Connect

Direct Connect (DX) provides a **dedicated physical connection** between your on-premises network and AWS — bypassing the public internet entirely.

### Connection speeds
- **Dedicated connections**: 1 Gbps, 10 Gbps, or 100 Gbps — a physical port at an AWS Direct Connect location
- **Hosted connections**: sub-1 Gbps (50 Mbps, 200 Mbps, etc.) or up to 10 Gbps — provisioned by an AWS Direct Connect Partner

### Virtual Interfaces (VIFs)
A single DX connection can carry multiple Virtual Interfaces:

| VIF Type | Accesses | Use case |
|---|---|---|
| **Private VIF** | VPC resources via private IPs | Connect to EC2, RDS, etc. in a VPC |
| **Public VIF** | AWS public services (S3, DynamoDB, CloudFront) | Access public AWS endpoints with consistent bandwidth |
| **Transit VIF** | Transit Gateway | Connect DX to many VPCs via TGW |

### Direct Connect Gateway
A **Direct Connect Gateway** lets you use **one DX connection to access VPCs in multiple regions**. Without it, you'd need a separate DX connection per region.

```
On-premises ── DX ── DX Gateway ──┬── VPC us-east-1
                                   ├── VPC eu-west-1
                                   └── VPC ap-southeast-1
```

### Key properties
- **Not encrypted by default** — DX is a private physical link but not cryptographically encrypted. Add a VPN on top of DX for encryption.
- **Consistent latency** — unlike VPN over internet, DX provides predictable, low-latency performance
- **Lead time**: provisioning a dedicated DX connection takes **weeks to months** (physical cable installation at a colocation facility)
- **No internet traversal** — useful for compliance requirements prohibiting data transfer over the public internet

### Resiliency
- For **maximum resiliency**: two DX connections from two different DX locations to two different VGWs
- For **high resiliency**: two DX connections at the same DX location
- For **development/backup**: one DX connection + a Site-to-Site VPN as failover

> **Exam tip:** DX = dedicated physical connection, consistent bandwidth, not encrypted by default, weeks to provision. VPN = quick to set up (minutes), encrypted, over internet, variable latency.

## Connectivity Decision Guide

```
Connect two VPCs?
├── Few VPCs, same or different account/region → VPC Peering
└── Many VPCs, transitive routing needed       → Transit Gateway

Access AWS services privately from VPC?
├── S3 or DynamoDB                             → Gateway Endpoint (free)
└── Other AWS services                         → Interface Endpoint (PrivateLink)

Expose a service in your VPC to other VPCs?
└── AWS PrivateLink (NLB + Interface Endpoint)

Connect on-premises to AWS?
├── Fast setup, acceptable internet latency    → Site-to-Site VPN
├── High bandwidth, consistent latency         → Direct Connect
├── Many on-premises sites to one AWS          → VPN CloudHub
├── DX + backup                               → DX primary + VPN failover
└── DX to many VPCs or regions               → DX Gateway (+ Transit VIF)

Need encryption on Direct Connect?
└── Run VPN tunnel over the DX connection
```

## Working with VPC Connectivity via boto3

In [ ]:
import boto3

ec2 = boto3.client('ec2', region_name='us-east-1')

In [ ]:
# Request a VPC peering connection
peering = ec2.create_vpc_peering_connection(
    VpcId='vpc-0aaa111',          # requester VPC
    PeerVpcId='vpc-0bbb222',      # accepter VPC
    PeerOwnerId='987654321098',   # accepter account (omit if same account)
    PeerRegion='us-west-2',       # omit if same region
    TagSpecifications=[{
        'ResourceType': 'vpc-peering-connection',
        'Tags': [{'Key': 'Name', 'Value': 'prod-to-shared-services'}]
    }]
)
pcx_id = peering['VpcPeeringConnection']['VpcPeeringConnectionId']
print(f"Peering connection requested: {pcx_id}")

# Accept (same account / same region only — cross-account requires accepter to call this)
ec2.accept_vpc_peering_connection(VpcPeeringConnectionId=pcx_id)

# Add route in requester VPC's route table
ec2.create_route(
    RouteTableId='rtb-0requester',
    DestinationCidrBlock='10.1.0.0/16',   # peer VPC CIDR
    VpcPeeringConnectionId=pcx_id
)
print("Route added — peering active")

In [ ]:
# Create a Gateway Endpoint for S3 (free, adds route to route table)
ep = ec2.create_vpc_endpoint(
    VpcId='vpc-0aaa111',
    ServiceName='com.amazonaws.us-east-1.s3',
    VpcEndpointType='Gateway',
    RouteTableIds=['rtb-0private1', 'rtb-0private2'],  # add to private route tables
    PolicyDocument='''{
        "Statement": [{
            "Effect": "Allow",
            "Principal": "*",
            "Action": "s3:GetObject",
            "Resource": "arn:aws:s3:::my-app-bucket/*"
        }]
    }'''  # endpoint policy restricts to one bucket
)
print(f"S3 Gateway Endpoint: {ep['VpcEndpoint']['VpcEndpointId']}")

In [ ]:
# Create an Interface Endpoint for SSM (Systems Manager)
# Allows private EC2 instances to use SSM Session Manager without NAT Gateway
ssm_ep = ec2.create_vpc_endpoint(
    VpcId='vpc-0aaa111',
    ServiceName='com.amazonaws.us-east-1.ssm',
    VpcEndpointType='Interface',
    SubnetIds=['subnet-0private1a', 'subnet-0private1b'],  # one per AZ
    SecurityGroupIds=['sg-0endpoint'],
    PrivateDnsEnabled=True   # standard DNS name resolves to private IP inside VPC
)
print(f"SSM Interface Endpoint: {ssm_ep['VpcEndpoint']['VpcEndpointId']}")

# Note: for full SSM access you also need endpoints for ssmmessages and ec2messages

In [ ]:
# Create a Transit Gateway and attach two VPCs
tgw = ec2.create_transit_gateway(
    Description='Central hub for all VPCs',
    Options={
        'DefaultRouteTableAssociation': 'enable',
        'DefaultRouteTablePropagation': 'enable',
        'AutoAcceptSharedAttachments': 'disable',
        'DnsSupport': 'enable'
    }
)
tgw_id = tgw['TransitGateway']['TransitGatewayId']
print(f"Transit Gateway: {tgw_id}")

# Attach VPC A to TGW (one subnet per AZ)
att_a = ec2.create_transit_gateway_vpc_attachment(
    TransitGatewayId=tgw_id,
    VpcId='vpc-0aaa111',
    SubnetIds=['subnet-0priv1a', 'subnet-0priv1b']
)
print(f"VPC A attached: {att_a['TransitGatewayVpcAttachment']['TransitGatewayAttachmentId']}")

In [ ]:
# Create a Site-to-Site VPN connection

# 1. Customer Gateway — represents on-premises VPN device
cgw = ec2.create_customer_gateway(
    Type='ipsec.1',
    IpAddress='203.0.113.50',  # on-premises public IP
    BgpAsn=65000               # on-premises ASN for BGP dynamic routing
)
cgw_id = cgw['CustomerGateway']['CustomerGatewayId']

# 2. Virtual Private Gateway — attach to VPC
vgw = ec2.create_vpn_gateway(Type='ipsec.1')
vgw_id = vgw['VpnGateway']['VpnGatewayId']
ec2.attach_vpn_gateway(VpnGatewayId=vgw_id, VpcId='vpc-0aaa111')

# 3. VPN Connection
vpn = ec2.create_vpn_connection(
    Type='ipsec.1',
    CustomerGatewayId=cgw_id,
    VpnGatewayId=vgw_id,
    Options={'StaticRoutesOnly': False}  # False = use BGP dynamic routing
)
print(f"VPN Connection: {vpn['VpnConnection']['VpnConnectionId']}")
print("Download customer gateway configuration from console to configure on-premises device")

## Summary

| Concept | Key Takeaway |
|---|---|
| VPC Peering | Direct VPC-to-VPC connection; non-transitive; no overlapping CIDRs; route tables must be updated |
| Gateway Endpoint | Free; S3 and DynamoDB only; adds route to route table |
| Interface Endpoint | Paid; most AWS services; ENI with private IP; supports private DNS |
| PrivateLink | Expose a service (not whole VPC) to consumers via NLB + Interface Endpoint |
| Transit Gateway | Hub-and-spoke for many VPCs; transitive routing; supports VPN + DX attachments |
| TGW route tables | Segment traffic — prevent dev VPCs from reaching prod VPCs |
| TGW RAM sharing | Share one TGW across multiple accounts |
| Site-to-Site VPN | IPsec tunnel; encrypted; over internet; up to 1.25 Gbps; minutes to set up |
| VPN — two tunnels | Always two tunnels per VPN connection for redundancy |
| VPN CloudHub | Multiple on-premises sites via one VGW; requires BGP |
| Direct Connect | Dedicated physical link; consistent latency; not encrypted by default; weeks to provision |
| DX — Private VIF | Access VPC resources via private IPs |
| DX — Public VIF | Access public AWS services (S3, DynamoDB) |
| DX — Transit VIF | Connect DX to Transit Gateway |
| Direct Connect Gateway | One DX connection to VPCs in multiple regions |
| DX + encryption | Run VPN over DX for encrypted dedicated connectivity |
| DX + VPN backup | DX as primary; Site-to-Site VPN as failover |